# **Advanced Machine Learning & MLOps - Tourism Package Prediction**

## Business Context

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement.

## Objective

As an MLOps Engineer at "Visit with Us," the objective is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary goal is to build a model that predicts whether a customer will purchase the newly introduced **Wellness Tourism Package** before contacting them. The pipeline includes data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system enables automated updates, streamlined model deployment, and improved operational efficiency.

## Data Description

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package.

**Customer Details**

| Feature | Description |
|---|---|
| `CustomerID` | Unique identifier for each customer |
| `ProdTaken` | **Target variable** - whether the customer purchased a package (0: No, 1: Yes) |
| `Age` | Age of the customer |
| `TypeofContact` | Method of contact (Company Invited or Self Inquiry) |
| `CityTier` | City category based on development (Tier 1 > Tier 2 > Tier 3) |
| `Occupation` | Customer's occupation |
| `Gender` | Gender of the customer |
| `NumberOfPersonVisiting` | Number of people accompanying the customer |
| `PreferredPropertyStar` | Preferred hotel star rating |
| `MaritalStatus` | Marital status |
| `NumberOfTrips` | Average annual trips |
| `Passport` | Whether the customer holds a passport (0: No, 1: Yes) |
| `OwnCar` | Whether the customer owns a car (0: No, 1: Yes) |
| `NumberOfChildrenVisiting` | Number of children below 5 accompanying |
| `Designation` | Designation in current organization |
| `MonthlyIncome` | Gross monthly income |

**Customer Interaction Data**

| Feature | Description |
|---|---|
| `PitchSatisfactionScore` | Satisfaction score for the sales pitch |
| `ProductPitched` | Type of product pitched |
| `NumberOfFollowups` | Follow-ups after the pitch |
| `DurationOfPitch` | Duration of the sales pitch |

## Pre-requisites

This project deploys everything on **GitHub Actions** (the ML pipeline) and **Streamlit Community Cloud** (the live app). You will need:

1. A **GitHub account** with a Personal Access Token (PAT) that has `repo` and `workflow` scopes.
2. A **Google Colab** environment (this notebook is designed to run end-to-end in Colab).
3. The `tourism.csv` dataset uploaded to the `tourism_project/data/` folder created below.

## Installing and Importing Necessary Libraries

In [ ]:
!pip install -q PyGithub==2.9.1
!pip install -q mlflow==3.0.1 pyngrok==7.2.12
!pip install -q xgboost==3.0.0 streamlit==1.45.1

**Note:** If the `pip install` cell above triggers a session restart prompt, restart the runtime and run all cells again from the top.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from github import Github, GithubException

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
import joblib
import mlflow

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## Configuration

In [ ]:
# Edit these three values, then run every cell top to bottom.
GITHUB_USERNAME   = "Rajkoti92"                           # Your GitHub username
REPO_NAME         = "tourism-package-mlops"               # Repository name
COLAB_SECRET_NAME = "GITHUB_TOKEN"                        # Name of the secret in Colab

REPO   = f"{GITHUB_USERNAME}/{REPO_NAME}"
BRANCH = "main"

### Secrets in Colab

**Secrets** in Google Colab provide a secure way to store sensitive information such as API keys and access tokens.

1. In the left pane of Google Colab, click the **key icon** (fifth icon).
2. Click **+ Add new secret**. In the **Name** field, enter `GITHUB_TOKEN`. In the **Value** field, paste your GitHub Personal Access Token.
3. Toggle **Notebook access** to ON.

In [ ]:
# Make the token available to both PyGithub and the gh CLI
import os
from google.colab import userdata

os.environ['GH_TOKEN'] = userdata.get(COLAB_SECRET_NAME)
print("Token loaded.")

# Model Building

In [ ]:
# Create the master folder and subfolders for the project
os.makedirs('tourism_project', exist_ok=True)
os.makedirs('tourism_project/model_building', exist_ok=True)
os.makedirs('tourism_project/data', exist_ok=True)
os.makedirs('tourism_project/deployment', exist_ok=True)
print("Project folders created.")

## Data Registration

Run the cell below to upload `tourism.csv` from your local machine into `tourism_project/data/`.

In [ ]:
from google.colab import files
import shutil

print("Select tourism.csv from your local machine:")
uploaded = files.upload()

for filename in uploaded:
    shutil.move(filename, 'tourism_project/data/tourism.csv')
    print(f"Saved {filename} -> tourism_project/data/tourism.csv")

In [ ]:
%%writefile tourism_project/model_building/data_register.py
"""Data Registration: validate the tourism dataset and print a summary."""
import pandas as pd
import sys

DATA_PATH = "tourism_project/data/tourism.csv"

EXPECTED_COLUMNS = [
    "CustomerID", "ProdTaken", "Age", "TypeofContact", "CityTier",
    "DurationOfPitch", "Occupation", "Gender", "NumberOfPersonVisiting",
    "NumberOfFollowups", "ProductPitched", "PreferredPropertyStar",
    "MaritalStatus", "NumberOfTrips", "Passport", "PitchSatisfactionScore",
    "OwnCar", "NumberOfChildrenVisiting", "Designation", "MonthlyIncome",
]

df = pd.read_csv(DATA_PATH)

# Drop unnamed index column if present
if df.columns[0].startswith("Unnamed"):
    df = df.drop(columns=df.columns[0])

# Validate columns
missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
if missing_cols:
    print(f"VALIDATION FAILED - missing columns: {missing_cols}")
    sys.exit(1)

print("VALIDATION PASSED")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\nTarget distribution (ProdTaken):\n{df['ProdTaken'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
!python tourism_project/model_building/data_register.py

## Exploratory Data Analysis

Before building the pipeline scripts, we explore the dataset to understand its characteristics and inform our data-cleaning and feature-engineering decisions.

### Data Overview

In [ ]:
# Load the dataset
data = pd.read_csv('tourism_project/data/tourism.csv')

# Drop unnamed index column if present
if data.columns[0].startswith('Unnamed'):
    data = data.drop(columns=data.columns[0])

df = data.copy()
df.head()

In [ ]:
# Shape and data types
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
print()
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).query('Missing > 0').sort_values('Missing', ascending=False)

In [ ]:
# Target distribution
print('Target variable (ProdTaken) distribution:')
print(df['ProdTaken'].value_counts())
print(f'\nPurchase rate: {df["ProdTaken"].mean():.2%}')

In [ ]:
# Statistical summary - numerical columns
df.describe().T

In [ ]:
# Statistical summary - categorical columns
df.describe(include='object').T

**Observations - Data Overview**

- The dataset has ~4,888 rows and 20 columns, with a binary target `ProdTaken`.
- Several columns have missing values: `Age`, `DurationOfPitch`, `NumberOfFollowups`, `PreferredPropertyStar`, `NumberOfTrips`, `NumberOfChildrenVisiting`, and `MonthlyIncome`.
- The `Gender` column contains the misspelled value `'Fe Male'` which should be `'Female'`.
- The target is imbalanced - significantly more non-purchasers (0) than purchasers (1), requiring careful model evaluation beyond accuracy.
- `CustomerID` is a unique identifier and carries no predictive signal.

### Univariate Analysis

In [ ]:
def histogram_boxplot(frame, feature, bins=30):
    """Histogram + boxplot for a numerical column."""
    fig, (ax_box, ax_hist) = plt.subplots(
        nrows=2, sharex=True, gridspec_kw={'height_ratios': (0.25, 0.75)}, figsize=(10, 5)
    )
    sns.boxplot(data=frame, x=feature, ax=ax_box, color='violet', showmeans=True)
    sns.histplot(data=frame, x=feature, bins=bins, ax=ax_hist, kde=True)
    ax_hist.axvline(frame[feature].mean(), color='green', linestyle='--', label='Mean')
    ax_hist.axvline(frame[feature].median(), color='black', linestyle='-', label='Median')
    ax_hist.legend()
    ax_box.set_title(f'Distribution of {feature}')
    plt.show()

In [ ]:
histogram_boxplot(df, 'Age')

In [ ]:
histogram_boxplot(df, 'DurationOfPitch')

In [ ]:
histogram_boxplot(df, 'MonthlyIncome')

In [ ]:
histogram_boxplot(df, 'NumberOfTrips')

In [ ]:
def labeled_barplot(frame, feature, rotation=0):
    """Labeled barplot for a categorical column."""
    plt.figure(figsize=(10, 4))
    ax = sns.countplot(data=frame, x=feature, order=frame[feature].value_counts().index, palette='Paired')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom')
    plt.xticks(rotation=rotation)
    plt.title(f'Count of observations by {feature}')
    plt.show()

In [ ]:
labeled_barplot(df, 'TypeofContact')

In [ ]:
labeled_barplot(df, 'Occupation')

In [ ]:
labeled_barplot(df, 'Gender')

In [ ]:
labeled_barplot(df, 'ProductPitched')

In [ ]:
labeled_barplot(df, 'MaritalStatus')

In [ ]:
labeled_barplot(df, 'Designation', rotation=15)

**Observations - Univariate Analysis**

- `Age` ranges from ~18 to ~60+, with a near-normal distribution centered around 35-40.
- `DurationOfPitch` is right-skewed; most pitches are short (under 15 min) but some extend to 30+.
- `MonthlyIncome` shows distinct clusters corresponding to designation levels.
- `Gender` confirms the `'Fe Male'` typo needing correction. The split is roughly balanced.
- `ProductPitched` is dominated by `Basic`, followed by `Deluxe` and `Standard`; premium tiers (`King`, `Super Deluxe`) are rare.
- Most customers are `Salaried` or `Small Business`; `Large Business` and `Free Lancer` are underrepresented.
- `Self Enquiry` is the dominant contact type.

### Bivariate Analysis

In [ ]:
# Purchase rate by product pitched
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('ProductPitched')['ProdTaken'].mean().sort_values(ascending=False).plot.bar(
    ax=axes[0], color='steelblue')
axes[0].set_title('Purchase Rate by Product Pitched')
axes[0].set_ylabel('Purchase Rate')
axes[0].tick_params(axis='x', rotation=15)

df.groupby('Designation')['ProdTaken'].mean().sort_values(ascending=False).plot.bar(
    ax=axes[1], color='coral')
axes[1].set_title('Purchase Rate by Designation')
axes[1].set_ylabel('Purchase Rate')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Purchase rate by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.ravel(), ['TypeofContact', 'MaritalStatus', 'Occupation', 'Gender']):
    df.groupby(col)['ProdTaken'].mean().sort_values(ascending=False).plot.bar(ax=ax, color='teal')
    ax.set_title(f'Purchase Rate by {col}')
    ax.set_ylabel('Purchase Rate')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numerical features with the target
num_cols = df.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(12, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='Spectral', vmin=-1, vmax=1)
plt.title('Correlation between numerical variables')
plt.show()

In [ ]:
# Distribution of numerical features by target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.ravel(), ['Age', 'DurationOfPitch', 'MonthlyIncome',
                                   'NumberOfFollowups', 'NumberOfTrips', 'NumberOfPersonVisiting']):
    sns.boxplot(data=df, x='ProdTaken', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} vs ProdTaken')
plt.tight_layout()
plt.show()

**Observations - Bivariate Analysis**

- **ProductPitched** is a strong predictor: `Basic` has a significantly higher purchase rate than premium products.
- **Passport** holders show a notably higher purchase rate - travel readiness signals buying intent.
- **Designation** and **MonthlyIncome** are correlated and both influence purchasing, with mid-level employees (Executives, Managers) being the primary buyers.
- **NumberOfFollowups** has a positive relationship with purchases - more follow-ups correlate with conversion.
- **MaritalStatus** shows variation, with single and unmarried customers being more likely to purchase.
- **CityTier** and the target have a weak relationship - city tier alone is not a strong differentiator.

**Key Insights from EDA**

1. The dataset has missing values in 7 columns - these must be imputed (median for numerical, mode for categorical) in the pipeline.
2. The `Gender` column has a `'Fe Male'` typo requiring correction.
3. `CustomerID` is a unique identifier with no predictive value and must be dropped.
4. The target is imbalanced (~18-19% purchase rate), so **Recall** and **F1-score** for the positive class are more important than raw accuracy.
5. Key predictive features include `ProductPitched`, `Passport`, `NumberOfFollowups`, `Designation`, `MaritalStatus`, and `Age`.

## Data Preparation

The script below loads the dataset, performs cleaning (fixing typos, imputing missing values, dropping the identifier column), and splits the data into training and testing sets.

In [ ]:
%%writefile tourism_project/model_building/prep.py
"""Data Preparation: clean the tourism dataset and create train/test splits."""
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "tourism_project/data/tourism.csv"

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Drop unnamed index column if present
if df.columns[0].startswith("Unnamed"):
    df = df.drop(columns=df.columns[0])

# --- Data Cleaning ---
# Drop the unique identifier (no predictive value)
df = df.drop(columns=["CustomerID"])

# Fix typo in Gender column
df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})

# Impute missing values
# Numerical columns: fill with median
num_cols = df.select_dtypes(include="number").columns.tolist()
num_cols.remove("ProdTaken")  # exclude target
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical columns: fill with mode
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Cleaned data shape: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

# --- Train-Test Split ---
X = df.drop(columns=["ProdTaken"])
y = df["ProdTaken"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Save the splits
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)
train_df.to_csv("tourism_project/data/train.csv", index=False)
test_df.to_csv("tourism_project/data/test.csv", index=False)

print(f"Train set: {train_df.shape[0]} rows, Test set: {test_df.shape[0]} rows")
print(f"Train purchase rate: {y_train.mean():.2%}")
print(f"Test purchase rate:  {y_test.mean():.2%}")

In [ ]:
!python tourism_project/model_building/prep.py

## Model Training and Registration with Experimentation Tracking

The script below loads the train and test splits, trains multiple models (Decision Tree, Random Forest, XGBoost), tunes the best-performing model with GridSearchCV, logs all experiments to MLflow, and saves the best model.

In [ ]:
%%writefile tourism_project/model_building/train.py
"""Model Training: train, tune, evaluate, log with MLflow, and save the best model."""
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
)
import joblib
import mlflow

# --- Load the train and test splits ---
train_df = pd.read_csv("tourism_project/data/train.csv")
test_df = pd.read_csv("tourism_project/data/test.csv")

TARGET = "ProdTaken"
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Identify column types
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()
print(f"Categorical: {cat_cols}")
print(f"Numerical:   {num_cols}")

# --- Preprocessing pipeline ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", drop="first")),
        ]), cat_cols),
    ]
)


def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    """Train and evaluate a pipeline, returning a metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred),
        "recall": recall_score(y_te, y_pred),
        "f1": f1_score(y_te, y_pred),
    }
    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_te, y_proba)
    return metrics


# --- Baseline model comparison ---
models = {
    "Decision Tree": Pipeline([("pre", preprocessor), ("clf", DecisionTreeClassifier(random_state=42))]),
    "Random Forest": Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("pre", preprocessor), ("clf", GradientBoostingClassifier(random_state=42))]),
    "XGBoost": Pipeline([("pre", preprocessor), ("clf", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss"))]),
}

mlflow.set_experiment("tourism-package-prediction")

results = {}
for name, pipe in models.items():
    with mlflow.start_run(run_name=f"baseline-{name}"):
        metrics = evaluate_model(pipe, X_train, y_train, X_test, y_test)
        results[name] = metrics
        mlflow.log_params({"model": name, "stage": "baseline"})
        mlflow.log_metrics(metrics)
        print(f"\n{name}:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        print(classification_report(y_test, pipe.predict(X_test)))

# Pick the best baseline model by F1 score
results_df = pd.DataFrame(results).T
print("\n=== Baseline Model Comparison ===")
print(results_df.round(4))
best_baseline = results_df["f1"].idxmax()
print(f"\nBest baseline model by F1: {best_baseline}")


# --- Hyperparameter Tuning (top 2 models: Random Forest and XGBoost) ---
print("\n=== Hyperparameter Tuning ===")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Tune Random Forest
rf_param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 12, None],
    "clf__min_samples_leaf": [2, 5],
}

rf_grid = GridSearchCV(
    Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
rf_grid.fit(X_train, y_train)
print(f"\nBest RF params: {rf_grid.best_params_}")
print(f"Best RF CV F1: {rf_grid.best_score_:.4f}")

with mlflow.start_run(run_name="tuned-random-forest"):
    rf_metrics = evaluate_model(rf_grid.best_estimator_, X_train, y_train, X_test, y_test)
    mlflow.log_params(rf_grid.best_params_)
    mlflow.log_params({"model": "Random Forest", "stage": "tuned"})
    mlflow.log_metrics(rf_metrics)
    print("Tuned Random Forest:")
    for k, v in rf_metrics.items():
        print(f"  {k}: {v:.4f}")
    print(classification_report(y_test, rf_grid.best_estimator_.predict(X_test)))


# Tune XGBoost
xgb_param_grid = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    Pipeline([("pre", preprocessor), ("clf", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss"))]),
    param_grid=xgb_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
xgb_grid.fit(X_train, y_train)
print(f"\nBest XGBoost params: {xgb_grid.best_params_}")
print(f"Best XGBoost CV F1: {xgb_grid.best_score_:.4f}")

with mlflow.start_run(run_name="tuned-xgboost"):
    xgb_metrics = evaluate_model(xgb_grid.best_estimator_, X_train, y_train, X_test, y_test)
    mlflow.log_params(xgb_grid.best_params_)
    mlflow.log_params({"model": "XGBoost", "stage": "tuned"})
    mlflow.log_metrics(xgb_metrics)
    print("Tuned XGBoost:")
    for k, v in xgb_metrics.items():
        print(f"  {k}: {v:.4f}")
    print(classification_report(y_test, xgb_grid.best_estimator_.predict(X_test)))


# --- Select and save the best overall model ---
all_tuned = {
    "Tuned Random Forest": (rf_grid.best_estimator_, rf_metrics),
    "Tuned XGBoost": (xgb_grid.best_estimator_, xgb_metrics),
}

best_name = max(all_tuned, key=lambda k: all_tuned[k][1]["f1"])
best_model = all_tuned[best_name][0]
best_metrics = all_tuned[best_name][1]

print(f"\n=== Final Model: {best_name} ===")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")

# Save the model for the deployment step
MODEL_PATH = "tourism_project/deployment/tourism_model.joblib"
joblib.dump(best_model, MODEL_PATH)
print(f"\nModel saved to {MODEL_PATH}")

In [ ]:
!python tourism_project/model_building/train.py

### Model Evaluation - Confusion Matrix and ROC Curve

The visualizations below use the model saved by the training script.

In [ ]:
# Load the saved model and evaluate visually
model = joblib.load('tourism_project/deployment/tourism_model.joblib')

test_df = pd.read_csv('tourism_project/data/test.csv')
X_test_viz = test_df.drop(columns=['ProdTaken'])
y_test_viz = test_df['ProdTaken']

y_pred = model.predict(X_test_viz)
y_proba = model.predict_proba(X_test_viz)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test_viz, y_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(y_test_viz, y_proba, ax=axes[1])
axes[1].set_title('ROC Curve')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test_viz, y_pred))

**Comments on Model Performance**

- The tuned model achieves strong classification performance with a good balance between precision and recall for the positive class.
- The ROC-AUC score indicates strong discriminatory power between purchasers and non-purchasers.
- The confusion matrix shows the model correctly identifies both classes, with manageable false positives and false negatives.
- This model is suitable for deployment to help the marketing team prioritize customer outreach.

### Export Model for Deployment (Portable Format)

The trained sklearn pipeline cannot be deployed directly on Streamlit Community Cloud because Streamlit Cloud uses a Python version that may be incompatible with the pickled sklearn objects. Instead, we export:

1. **`xgboost_model.json`** — the XGBoost model in native JSON format (version-independent, no pickle).
2. **`preprocessing_config.json`** — the preprocessing parameters (imputation values, categories) extracted from the fitted sklearn ColumnTransformer.

The Streamlit app reimplements the preprocessing in pure pandas and loads the model via `xgb.Booster`, eliminating the sklearn dependency entirely at inference time.

In [ ]:
import json

# Load the trained sklearn pipeline
model = joblib.load('tourism_project/deployment/tourism_model.joblib')

# --- Extract preprocessing parameters from the fitted ColumnTransformer ---
preprocessor = model.named_steps["pre"]

# Numeric transformer: SimpleImputer with median
num_transformer = preprocessor.transformers_[0]  # ("num", SimpleImputer, num_cols)
num_imputer = num_transformer[1]
num_columns = num_transformer[2]
num_fill_values = {col: float(val) for col, val in zip(num_columns, num_imputer.statistics_)}

# Categorical transformer: Pipeline(SimpleImputer -> OneHotEncoder)
cat_transformer = preprocessor.transformers_[1]  # ("cat", Pipeline, cat_cols)
cat_pipeline = cat_transformer[1]
cat_columns = cat_transformer[2]
cat_imputer = cat_pipeline.named_steps["impute"]
cat_encoder = cat_pipeline.named_steps["encode"]
cat_fill_values = {col: str(val) for col, val in zip(cat_columns, cat_imputer.statistics_)}
categories = {col: [str(c) for c in cats] for col, cats in zip(cat_columns, cat_encoder.categories_)}

# Build the feature name list matching sklearn's output order
feature_names_out = []
for col in num_columns:
    feature_names_out.append(f"num__{col}")
for col in cat_columns:
    cats = categories[col]
    for cat in cats[1:]:  # drop='first' skips the first category
        feature_names_out.append(f"cat__{col}_{cat}")

preprocessing_config = {
    "numeric_columns": list(num_columns),
    "numeric_fill_values": num_fill_values,
    "categorical_columns": list(cat_columns),
    "categorical_fill_values": cat_fill_values,
    "categories": categories,
    "feature_names_out": feature_names_out,
}

# Save preprocessing config
config_path = "tourism_project/deployment/preprocessing_config.json"
with open(config_path, "w") as f:
    json.dump(preprocessing_config, f, indent=2)
print(f"Preprocessing config saved to {config_path}")
print(f"Total features: {len(feature_names_out)}")

# --- Export XGBoost model in native JSON format ---
xgb_model = model.named_steps["clf"]
model_path = "tourism_project/deployment/xgboost_model.json"
xgb_model.save_model(model_path)
print(f"XGBoost model saved to {model_path}")

# --- Verify portable model matches original pipeline ---
import xgboost as xgb

portable_model = xgb.Booster()
portable_model.load_model(model_path)

with open(config_path) as f:
    config = json.load(f)

def preprocess_portable(df):
    """Replicate sklearn preprocessing in pure pandas."""
    result = pd.DataFrame()
    for col in config["numeric_columns"]:
        result[f"num__{col}"] = df[col].fillna(config["numeric_fill_values"][col])
    for col in config["categorical_columns"]:
        filled = df[col].fillna(config["categorical_fill_values"][col])
        cats = config["categories"][col]
        for cat in cats[1:]:
            result[f"cat__{col}_{cat}"] = (filled == cat).astype(float)
    return result[config["feature_names_out"]].values.astype(np.float32)

# Test with 3 sample inputs
test_cases = [
    {"Age": 35, "TypeofContact": "Self Enquiry", "CityTier": 1, "DurationOfPitch": 10,
     "Occupation": "Salaried", "Gender": "Male", "NumberOfPersonVisiting": 2,
     "NumberOfFollowups": 3.0, "ProductPitched": "Basic", "PreferredPropertyStar": 3.0,
     "MaritalStatus": "Single", "NumberOfTrips": 2.0, "Passport": 0,
     "PitchSatisfactionScore": 3, "OwnCar": 0, "NumberOfChildrenVisiting": 0.0,
     "Designation": "Executive", "MonthlyIncome": 20000},
    {"Age": 55, "TypeofContact": "Company Invited", "CityTier": 3, "DurationOfPitch": 30,
     "Occupation": "Free Lancer", "Gender": "Female", "NumberOfPersonVisiting": 4,
     "NumberOfFollowups": 5.0, "ProductPitched": "King", "PreferredPropertyStar": 5.0,
     "MaritalStatus": "Married", "NumberOfTrips": 7.0, "Passport": 1,
     "PitchSatisfactionScore": 5, "OwnCar": 1, "NumberOfChildrenVisiting": 2.0,
     "Designation": "VP", "MonthlyIncome": 50000},
    {"Age": 28, "TypeofContact": "Self Enquiry", "CityTier": 2, "DurationOfPitch": 5,
     "Occupation": "Small Business", "Gender": "Male", "NumberOfPersonVisiting": 1,
     "NumberOfFollowups": 1.0, "ProductPitched": "Standard", "PreferredPropertyStar": 4.0,
     "MaritalStatus": "Unmarried", "NumberOfTrips": 1.0, "Passport": 0,
     "PitchSatisfactionScore": 2, "OwnCar": 0, "NumberOfChildrenVisiting": 0.0,
     "Designation": "Manager", "MonthlyIncome": 15000},
]

print("\nVerification - Portable model vs original pipeline:")
for i, tc in enumerate(test_cases):
    df_tc = pd.DataFrame([tc])
    orig_pred = model.predict(df_tc)[0]
    orig_proba = model.predict_proba(df_tc)[0]
    features = preprocess_portable(df_tc)
    dmatrix = xgb.DMatrix(features, feature_names=config["feature_names_out"])
    port_proba_pos = portable_model.predict(dmatrix)[0]
    port_pred = int(port_proba_pos > 0.5)
    match = orig_pred == port_pred
    print(f"  Case {i+1}: orig={orig_pred} (p={orig_proba[1]:.4f}), portable={port_pred} (p={port_proba_pos:.4f}) -> {'MATCH' if match else 'MISMATCH'}")

## Streamlit App

The app uses the portable deployment artifacts (`xgboost_model.json` and `preprocessing_config.json`) to run predictions without sklearn. It loads the XGBoost model via `xgb.Booster` (the pure C++ API) and reimplements the preprocessing in pandas, eliminating all pickle and sklearn dependencies at inference time.

In [ ]:
%%writefile tourism_project/deployment/app.py
"""Streamlit app for Tourism Package Purchase Prediction."""
import streamlit as st
import pandas as pd
import numpy as np
import xgboost as xgb
import os
import json

st.set_page_config(page_title="Tourism Package Predictor", page_icon="✈️", layout="centered")

BASE_DIR = os.path.dirname(__file__)

# Load preprocessing config and XGBoost model using pure Booster (no sklearn needed)
with open(os.path.join(BASE_DIR, "preprocessing_config.json")) as f:
    PREPROCESS_CONFIG = json.load(f)

MODEL = xgb.Booster()
MODEL.load_model(os.path.join(BASE_DIR, "xgboost_model.json"))

NUM_COLS = PREPROCESS_CONFIG["numeric_columns"]
NUM_FILL = PREPROCESS_CONFIG["numeric_fill_values"]
CAT_COLS = PREPROCESS_CONFIG["categorical_columns"]
CAT_FILL = PREPROCESS_CONFIG["categorical_fill_values"]
CAT_CATEGORIES = PREPROCESS_CONFIG["categories"]
FEATURE_ORDER = PREPROCESS_CONFIG["feature_names_out"]


def preprocess(df):
    """Replicate the sklearn ColumnTransformer preprocessing in pure pandas."""
    result = pd.DataFrame()
    # Numeric: fill missing with median, keep as-is
    for col in NUM_COLS:
        result[f"num__{col}"] = df[col].fillna(NUM_FILL[col])
    # Categorical: fill missing with mode, then one-hot encode (drop='first')
    for col in CAT_COLS:
        filled = df[col].fillna(CAT_FILL[col])
        categories = CAT_CATEGORIES[col]
        # drop='first' means skip the first category
        for cat in categories[1:]:
            result[f"cat__{col}_{cat}"] = (filled == cat).astype(float)
    return result[FEATURE_ORDER].values.astype(np.float32)


st.title("✈️ Tourism Package Purchase Predictor")
st.write("Predict whether a customer will purchase the Wellness Tourism Package.")

st.subheader("Customer Details")
col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=35)
    type_of_contact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    num_persons = st.number_input("Number of Persons Visiting", min_value=1, max_value=5, value=2)
    preferred_star = st.selectbox("Preferred Property Star", [3, 4, 5])
    marital_status = st.selectbox("Marital Status", ["Single", "Married", "Divorced", "Unmarried"])

with col2:
    num_trips = st.number_input("Number of Trips (annual)", min_value=1, max_value=10, value=2)
    passport = st.selectbox("Passport", [0, 1], format_func=lambda x: "Yes" if x else "No")
    pitch_score = st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5])
    own_car = st.selectbox("Own Car", [0, 1], format_func=lambda x: "Yes" if x else "No")
    num_children = st.number_input("Number of Children Visiting", min_value=0, max_value=5, value=0)
    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    monthly_income = st.number_input("Monthly Income", min_value=5000, max_value=100000, value=20000)

st.subheader("Interaction Details")
col3, col4 = st.columns(2)

with col3:
    duration_of_pitch = st.number_input("Duration of Pitch (min)", min_value=1, max_value=60, value=10)
    num_followups = st.number_input("Number of Follow-ups", min_value=1, max_value=6, value=3)

with col4:
    product_pitched = st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])

input_data = pd.DataFrame([{
    "Age": age,
    "TypeofContact": type_of_contact,
    "CityTier": city_tier,
    "DurationOfPitch": duration_of_pitch,
    "Occupation": occupation,
    "Gender": gender,
    "NumberOfPersonVisiting": num_persons,
    "NumberOfFollowups": float(num_followups),
    "ProductPitched": product_pitched,
    "PreferredPropertyStar": float(preferred_star),
    "MaritalStatus": marital_status,
    "NumberOfTrips": float(num_trips),
    "Passport": passport,
    "PitchSatisfactionScore": pitch_score,
    "OwnCar": own_car,
    "NumberOfChildrenVisiting": float(num_children),
    "Designation": designation,
    "MonthlyIncome": monthly_income,
}])

if st.button("Predict", type="primary"):
    features = preprocess(input_data)
    dmatrix = xgb.DMatrix(features, feature_names=FEATURE_ORDER)
    prob_positive = MODEL.predict(dmatrix)[0]
    prediction = int(prob_positive > 0.5)
    if prediction == 1:
        st.success(f"The customer is **likely to purchase** the package. (Confidence: {prob_positive:.1%})")
    else:
        st.info(f"The customer is **unlikely to purchase** the package. (Confidence: {1 - prob_positive:.1%})")

## App Dependencies

In [ ]:
%%writefile tourism_project/deployment/requirements.txt
streamlit
pandas
xgboost
numpy

In [ ]:
os.makedirs('.github/workflows', exist_ok=True)

In [ ]:
%%writefile .github/workflows/pipeline.yml
name: Tourism Package Prediction Pipeline

on:
  push:
    branches:
      - main
  workflow_dispatch:

permissions:
  contents: write

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r requirements_pipeline.txt
      - name: Register Dataset
        run: python tourism_project/model_building/data_register.py
      - name: Upload Registered Dataset
        uses: actions/upload-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data/tourism.csv

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r requirements_pipeline.txt
      - name: Download Registered Data
        uses: actions/download-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data/
      - name: Run Data Preparation
        run: python tourism_project/model_building/prep.py
      - name: Upload Train/Test Splits
        uses: actions/upload-artifact@v4
        with:
          name: train-test-data
          path: |
            tourism_project/data/train.csv
            tourism_project/data/test.csv

  train-model:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r requirements_pipeline.txt
      - name: Download Train/Test Splits
        uses: actions/download-artifact@v4
        with:
          name: train-test-data
          path: tourism_project/data/
      - name: Train Model
        run: python tourism_project/model_building/train.py
      - name: Commit Trained Model
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add tourism_project/deployment/tourism_model.joblib
          git diff --cached --quiet || git commit -m "Update trained model [skip ci]"
          git push

## Requirements File for the GitHub Actions Workflow

In [ ]:
%%writefile requirements_pipeline.txt
pandas==2.2.2
scikit-learn==1.6.1
xgboost==3.0.0
joblib==1.4.2
mlflow==3.0.1

## GitHub Authentication and Push Files

In [ ]:
# --- Safety check: Verify that the dataset exists before pushing ---
CSV_LOCAL = "tourism_project/data/tourism.csv"

assert os.path.exists(CSV_LOCAL), (
    f"{CSV_LOCAL} not found. Please upload tourism.csv into the "
    "tourism_project/data/ folder before running this cell."
)

# Connect to GitHub
gh = Github(os.environ["GH_TOKEN"])
user = gh.get_user()
REPO = f"{user.login}/{REPO_NAME}"

# Create the repository if it does not already exist
try:
    repo = gh.get_repo(REPO)
    print("Repository already exists:", repo.full_name)
except GithubException:
    repo = user.create_repo(
        REPO_NAME,
        private=False,
        auto_init=True
    )
    print("Repository created:", repo.full_name)

import base64, time


def push_file(local_path, repo_path, message):
    """Push a local file to the GitHub repository."""
    with open(local_path, "rb") as f:
        content = f.read()
    try:
        existing = repo.get_contents(repo_path, ref=BRANCH)
        repo.update_file(repo_path, message, content, existing.sha, branch=BRANCH)
        print(f"  Updated: {repo_path}")
    except GithubException:
        repo.create_file(repo_path, message, content, branch=BRANCH)
        print(f"  Created: {repo_path}")


# List of files to push
files_to_push = [
    ("tourism_project/data/tourism.csv", "tourism_project/data/tourism.csv"),
    ("tourism_project/model_building/data_register.py", "tourism_project/model_building/data_register.py"),
    ("tourism_project/model_building/prep.py", "tourism_project/model_building/prep.py"),
    ("tourism_project/model_building/train.py", "tourism_project/model_building/train.py"),
    ("tourism_project/deployment/app.py", "tourism_project/deployment/app.py"),
    ("tourism_project/deployment/requirements.txt", "tourism_project/deployment/requirements.txt"),
    ("tourism_project/deployment/tourism_model.joblib", "tourism_project/deployment/tourism_model.joblib"),
    ("tourism_project/deployment/preprocessing_config.json", "tourism_project/deployment/preprocessing_config.json"),
    ("tourism_project/deployment/xgboost_model.json", "tourism_project/deployment/xgboost_model.json"),
    (".github/workflows/pipeline.yml", ".github/workflows/pipeline.yml"),
    ("requirements_pipeline.txt", "requirements_pipeline.txt"),
]

print(f"Pushing {len(files_to_push)} files to {repo.full_name}...")
for local_path, repo_path in files_to_push:
    push_file(local_path, repo_path, f"Add {repo_path}")
    time.sleep(1)  # avoid rate limiting

print("\nAll files pushed successfully!")
print(f"Repository: https://github.com/{repo.full_name}")

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [ ]:
# GitHub Repository: https://github.com/Rajkoti92/tourism-package-mlops
# The repository contains the complete MLOps pipeline with:
#   - tourism_project/data/tourism.csv
#   - tourism_project/model_building/ (data_register.py, prep.py, train.py)
#   - tourism_project/deployment/ (app.py, requirements.txt, preprocessing_config.json, xgboost_model.json)
#   - .github/workflows/pipeline.yml
#   - requirements_pipeline.txt
# GitHub Actions workflow executed successfully (3 jobs: register-dataset -> data-prep -> train-model)
# The app uses xgb.Booster (native JSON format) with pure pandas preprocessing — no sklearn at inference time.

- Streamlit Community Cloud (link to the Streamlit app, screenshot of Streamlit app)

In [ ]:
# Streamlit App: https://tourism-package-mlops-oga2mxibcpsh5badzy9x8a.streamlit.app/
# The app collects 18 customer features and predicts Wellness Tourism Package purchase probability.
# Deployed on Streamlit Community Cloud from the Rajkoti92/tourism-package-mlops repository.

# Actionable Insights and Business Recommendations

## Insights from the Analysis

1. **Product Pitched is the strongest predictor.** Customers pitched `Basic` packages show a significantly higher purchase rate than those offered premium tiers. This suggests that affordability and value perception drive conversions more than luxury positioning.

2. **Passport ownership signals buying intent.** Customers with a valid passport are substantially more likely to purchase - they are travel-ready and represent a pre-qualified audience for international tourism packages.

3. **Follow-ups matter.** More sales follow-ups correlate with higher conversion rates. There is a sweet spot (around 4-5 follow-ups) beyond which marginal returns diminish.

4. **Mid-level professionals are the core buyers.** Executives and Managers with monthly incomes in the 17,000-25,000 range form the bulk of purchasers. Very senior (VP, AVP) and very junior designations convert less.

5. **Marital status influences purchase behavior.** Single and unmarried customers are more likely to purchase, potentially due to greater schedule flexibility and disposable income allocation toward travel.

6. **The target is imbalanced (~19% purchase rate).** The model's recall for the positive class is critical - missing a potential buyer has a higher business cost than a false positive (a wasted call).

## Business Recommendations

1. **Use the deployed prediction model to prioritize outreach.** Before each campaign wave, score the customer list through the Streamlit app or API, and prioritize contacts with the highest predicted purchase probability. This replaces the inconsistent manual targeting process.

2. **Lead with Basic packages.** Since `Basic` drives the highest conversion, use it as the entry-level offer. Upsell to Deluxe or Standard only after initial engagement - this matches the data's signal that affordability converts.

3. **Target passport holders in Tier 1-2 cities.** These customers combine travel readiness with higher per-customer revenue potential. Segment marketing spend accordingly.

4. **Optimize the follow-up cadence.** Ensure every pitched customer receives at least 3-4 follow-ups. Train the sales team to concentrate follow-up effort on customers the model scores highly.

5. **Retrain quarterly and monitor via the MLOps pipeline.** The GitHub Actions pipeline automatically retrains the model when new data is pushed. Monitor the F1-score and recall on each run to catch model drift early.

6. **Expand the feature set over time.** Consider adding features like seasonal travel trends, customer web-browsing behavior, and past booking history to improve the model's predictive power in future iterations.

**Power Ahead!**